Description: Run Cosmos T1 GGUF chat in Colab with cached installs and Drive model reuse.

# Run Cosmos T1 GGUF

Execution order:
1. Mount Drive
2. Install `llama-cpp-python` (from Drive wheel if available; fallback compile)
3. Run chat script
4. (Optional, one-time) build and save a wheel to Drive for faster future sessions


In [ ]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)


In [ ]:
##build and save wheel

# %%bash

# set -euo pipefail

# if [ -d /content/drive/MyDrive/wheels ]; then
  # WHEEL_DIR='/content/drive/MyDrive/wheels'
# elif [ -d '/content/drive/My Drive/wheels' ]; then
  # WHEEL_DIR='/content/drive/My Drive/wheels'
# else
  # WHEEL_DIR='/content/drive/MyDrive/wheels'
# fi
# mkdir -p "$WHEEL_DIR"

# echo 'Building wheel (one-time, slow) and saving to Drive cache...'
# CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75" pip wheel llama-cpp-python -w "$WHEEL_DIR"
# echo "Saved wheels under: $WHEEL_DIR"


In [ ]:
%%bash
#build and save llama-cpp-python wheel if does not exist + pip install from the wheel
set -euo pipefail

if [ -d /content/drive/MyDrive/wheels ]; then
  WHEEL_DIR='/content/drive/MyDrive/wheels'
elif [ -d '/content/drive/My Drive/wheels' ]; then
  WHEEL_DIR='/content/drive/My Drive/wheels'
else
  WHEEL_DIR='/content/drive/MyDrive/wheels'
  mkdir -p "$WHEEL_DIR"
fi

if ls "$WHEEL_DIR"/llama_cpp_python-*.whl >/dev/null 2>&1; then
  echo 'Installing llama-cpp-python from Drive wheel cache...'
  pip install -U "$WHEEL_DIR"/llama_cpp_python-*.whl
else
  echo 'No cached wheel found. Building/installing from source (slow)...'
  CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --no-cache-dir -U llama-cpp-python
fi


In [ ]:
# Run GGUF chat
from pathlib import Path
import runpy
import sys

if Path('/content/drive/MyDrive/training-embedding').exists():
    project_root = Path('/content/drive/MyDrive/training-embedding')
elif Path('/content/drive/My Drive/training-embedding').exists():
    project_root = Path('/content/drive/My Drive/training-embedding')
else:
    raise FileNotFoundError('Could not find training-embedding under Drive root.')

script_path = project_root / 'run_cosmos_t1_gguf.py'
if not script_path.exists():
    raise FileNotFoundError(f'Missing script: {script_path}')

# Drive-local GGUF cache path to avoid downloading each Colab restart.
save_load_path = project_root / 'models' / 'gguf_cache'
save_load_path.mkdir(parents=True, exist_ok=True)

print(f'Running: {script_path}')
print(f'Using --save_load_path: {save_load_path}')
print('Using --use_gpu')

old_argv = sys.argv[:]
try:
    sys.argv = [
        str(script_path),
        '--save_load_path', str(save_load_path),
        '--use_gpu',
    ]
    runpy.run_path(str(script_path), run_name='__main__')
finally:
    sys.argv = old_argv


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text = widgets.Text(
    placeholder="Type something here...",
    description="Input:"
)
button = widgets.Button(description="Submit")
output = widgets.Output()

def on_submit(_):
    with output:
        clear_output()
        print("You typed:", text.value)

button.on_click(on_submit)

display(text, button, output)


Text(value='', description='Input:', placeholder='Type something here...')

Button(description='Submit', style=ButtonStyle())

Output()